# Identificação do projeto

**Título do projeto:** Dengue no Brasil em 2026: pico de casos, gravidade e perfil dos pacientes

**Integrantes do grupo:** Bruna da Silva Carnelossi, João Pedro de Souza Quintiliano da Silva, João Victor Carvalho de Freitas, Matheus Henrique Heinzen

**Turma:** Sistemas de Informação • 6º período • Turma A GCL1 • Profª Marina de Lara

**Descrição breve do problema ou contexto:** a partir das notificações de dengue de 2026 preparadas no RA1, este notebook simula a chegada contínua de novas notificações e acompanha, em janelas de tempo, quantos casos e quantos casos graves surgem em cada região do país.

Repositório: https://github.com/JoaoVictorCarvalho07/api-dados-abertos-saude

---

# RA2: Streaming Data Processing and Time Windows

Utilizar os dados e o contexto do projeto para demonstrar o processamento em streaming e a aplicação de janelas temporais. Escolha um dos cenários:

- **Cenário A, dados estáticos:** simule a chegada de novos registros utilizando dados artificiais coerentes com o contexto do projeto.
- **Cenário B, dados em streaming:** utilize sua própria fonte de dados contínuos, sem necessidade de simulação.

```text
Notificações simuladas (perfil de casos reais do RA1)
        ↓
Arquivos JSON publicados aos poucos
        ↓
Pasta monitorada pelo Spark (readStream)
        ↓
Janela tumbling de 1 dia × região
        ↓
Tabela de resultado atualizada a cada lote
```

## 8. Objetivo da análise em streaming

- O que desejamos monitorar ou analisar?
- Qual informação espera obter com o processamento em streaming?

O objetivo da análise é **monitorar, dia a dia e por região, quantas novas notificações de dengue chegam e quantas delas são casos graves** (dengue com sinais de alarme ou dengue grave).

É a versão contínua das perguntas do RA1: em vez de olhar 2026 inteiro depois que tudo aconteceu (onde e quando houve mais casos, P1; quanto da carga é de casos graves, P2 e P3), o streaming mostra esses números **se formando enquanto as notificações chegam**. Uma secretaria de saúde usaria esse acompanhamento para perceber cedo uma região em que os casos, ou os casos graves, começam a subir.

## 9. Fonte de dados e processamento

- Informar o cenário escolhido: dados reais em streaming ou simulação.
- Explicar brevemente como os novos registros chegam.
- Implementearo processamento em tempo real.
- Demonstrar o recebimento de novos registros.

**Cenário A: simulação.** A base do projeto é um arquivo anual estático, que vai até 04/10/2026. Simulamos as notificações que chegariam **nos dias seguintes (05 a 07/10)**.

**Como os registros são criados.** Cada notificação simulada recebe um identificador próprio e um horário artificial dentro do dia, mas o **perfil do caso** (UF e região de residência, classificação final e se é grave) é sorteado entre casos reais do conjunto preparado no RA1. Assim, os registros são artificiais, mas mantêm as proporções reais entre regiões e entre casos leves e graves. O sorteio usa semente fixa: toda execução gera exatamente os mesmos registros.

**Como os registros chegam.** As notificações são agrupadas em **quatro lotes**, publicados um de cada vez como arquivos JSON numa pasta que o Spark monitora com `readStream`. Cada arquivo é escrito primeiro numa pasta de publicação e só depois movido para a pasta monitorada, para que o Spark nunca leia um arquivo pela metade. O último lote é **atrasado**: traz notificações do dia 05/10 que só chegam depois das dos dias 06 e 07.

### 9.1 Preparação do Spark

O Spark roda nesta máquina (`local[2]`), com duas partições nas operações de agrupamento, como nos exemplos de aula.

O fuso da sessão é **UTC** de propósito. As datas das notificações não carregam fuso; com o fuso de São Paulo, o Spark alinharia as janelas de 1 dia às 21:00 (meia-noite em UTC convertida para o horário de Brasília). Com UTC, a janela "05/10" vai de 00:00 a 24:00 do dia 05, exatamente como o dado foi registrado.

In [15]:
import json
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, date_format, sum as soma, window
from pyspark.sql.types import BooleanType, StringType, StructField, StructType, TimestampType

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

RAIZ = Path.cwd().parent
ARQUIVO_PREPARADO = RAIZ / "dados" / "preparados" / "dengue_brasil_2026.parquet"
assert ARQUIVO_PREPARADO.exists(), "Execute antes o notebook do RA1 (ra1/ra1_dengue_2026.ipynb), que gera o conjunto preparado."

spark = (
    SparkSession.builder.master("local[2]")
    .appName("Dengue 2026: notificações em streaming")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

area_temporaria = tempfile.TemporaryDirectory(prefix="dengue_streaming_")
pasta_base = Path(area_temporaria.name)
pasta_entrada = pasta_base / "entrada"
pasta_publicacao = pasta_base / "publicacao"
pasta_checkpoint = pasta_base / "checkpoint"
pasta_entrada.mkdir()
pasta_publicacao.mkdir()

print(f"✓ Spark {spark.version} iniciado")
print(f"Fuso da sessão:       {spark.conf.get('spark.sql.session.timeZone')}")
print(f"Arquivos na entrada:  {len(list(pasta_entrada.iterdir()))}")

✓ Spark 4.0.3 iniciado
Fuso da sessão:       UTC
Arquivos na entrada:  0


### 9.2 Perfil real usado na simulação

Os perfis são sorteados entre os casos com classificação final de dengue (10, 11 e 12) do conjunto preparado no RA1, os mesmos que formam o denominador de gravidade das perguntas P2 e P3.

In [16]:
referencia = pd.read_parquet(
    ARQUIVO_PREPARADO,
    columns=["uf_residencia", "regiao_residencia", "classificacao_final", "caso_grave"],
)
casos_classificados = referencia[referencia["caso_grave"].notna()].reset_index(drop=True)

print("BASE DE REFERÊNCIA (RA1)")
print("─" * 50)
print(f"Casos com classificação de dengue:  {len(casos_classificados):,}".replace(",", "."))
print(f"Proporção de casos graves:          {casos_classificados['caso_grave'].mean():.1%}")
print("\nParticipação de cada região nos casos:")
display(
    casos_classificados["regiao_residencia"].value_counts(normalize=True)
    .rename("participação").to_frame().style.format("{:.1%}")
)

BASE DE REFERÊNCIA (RA1)
──────────────────────────────────────────────────
Casos com classificação de dengue:  365.519
Proporção de casos graves:          3.5%

Participação de cada região nos casos:


,participação
regiao_residencia,
Sudeste,35.2%
Centro-Oeste,33.5%
Nordeste,22.1%
Norte,6.4%
Sul,2.8%


### 9.3 Gerando as notificações simuladas

Cada lote informa quantas notificações de cada dia ele traz. A função sorteia os perfis reais, cria horários artificiais dentro do dia e devolve as notificações já no formato do evento que chegará ao Spark.

In [17]:
gerador = np.random.default_rng(2026)

LOTES = [
    {"lote": 1, "chegada": "1ª", "dias": {"2026-10-05": 30}},
    {"lote": 2, "chegada": "2ª", "dias": {"2026-10-05": 20, "2026-10-06": 25}},
    {"lote": 3, "chegada": "3ª", "dias": {"2026-10-06": 15, "2026-10-07": 35}},
    {"lote": 4, "chegada": "4ª (atrasada)", "dias": {"2026-10-05": 10}},
]


def gerar_notificacoes(dia, quantidade, primeiro_id):
    perfis = casos_classificados.iloc[gerador.choice(len(casos_classificados), size=quantidade, replace=False)]
    minutos = np.sort(gerador.integers(0, 24 * 60, size=quantidade))
    horarios = pd.Timestamp(dia) + pd.to_timedelta(minutos, unit="min")
    return [
        {
            "id_notificacao": f"SIM-{primeiro_id + posicao:04d}",
            "dt_notificacao": horario.strftime("%Y-%m-%d %H:%M:%S"),
            "uf_residencia": perfil.uf_residencia,
            "regiao_residencia": perfil.regiao_residencia,
            "classificacao_final": perfil.classificacao_final,
            "caso_grave": bool(perfil.caso_grave),
        }
        for posicao, (horario, perfil) in enumerate(zip(horarios, perfis.itertuples()))
    ]


eventos_por_lote = {}
proximo_id = 1
for lote in LOTES:
    eventos = []
    for dia, quantidade in lote["dias"].items():
        eventos += gerar_notificacoes(dia, quantidade, proximo_id)
        proximo_id += quantidade
    eventos_por_lote[lote["lote"]] = eventos

resumo_lotes = pd.DataFrame(
    [
        {
            "lote": lote["lote"],
            "chegada": lote["chegada"],
            "dias das notificações": ", ".join(pd.Timestamp(dia).strftime("%d/%m") for dia in lote["dias"]),
            "notificações": len(eventos_por_lote[lote["lote"]]),
            "casos graves": sum(evento["caso_grave"] for evento in eventos_por_lote[lote["lote"]]),
        }
        for lote in LOTES
    ]
)
print("LOTES QUE VÃO CHEGAR")
display(resumo_lotes.style.hide(axis="index"))
print("Exemplo de notificação simulada (é este o dado que chega ao Spark):")
print(json.dumps(eventos_por_lote[1][0], ensure_ascii=False, indent=2))

LOTES QUE VÃO CHEGAR


lote,chegada,dias das notificações,notificações,casos graves
1,1ª,05/10,30,1
2,2ª,"05/10, 06/10",45,1
3,3ª,"06/10, 07/10",50,1
4,4ª (atrasada),05/10,10,0


Exemplo de notificação simulada (é este o dado que chega ao Spark):
{
  "id_notificacao": "SIM-0001",
  "dt_notificacao": "2026-10-05 00:05:00",
  "uf_residencia": "MT",
  "regiao_residencia": "Centro-Oeste",
  "classificacao_final": "Dengue",
  "caso_grave": false
}


### 9.4 Publicando um lote

Esta função representa o sistema de vigilância enviando um novo lote de notificações: grava um JSON com uma notificação por linha e o move para a pasta monitorada. Por enquanto, apenas definimos a função.

In [18]:
def publicar_lote(numero):
    nome_arquivo = f"lote_{numero:02d}.json"
    arquivo = pasta_publicacao / nome_arquivo
    with arquivo.open("w", encoding="utf-8") as saida:
        for notificacao in eventos_por_lote[numero]:
            saida.write(json.dumps(notificacao, ensure_ascii=False) + "\n")
    arquivo.replace(pasta_entrada / nome_arquivo)
    return len(eventos_por_lote[numero])

### 9.5 Schema e DataFrame de streaming

Como os dados chegam aos poucos, o Spark não pode descobrir os tipos olhando os arquivos: o **schema** é declarado antes. `dt_notificacao` é lido como data e hora, porque é o **tempo do evento** usado pelas janelas.

`spark.readStream` cria um DataFrame que existe mesmo com a pasta vazia e continua recebendo linhas novas.

In [19]:
esquema_notificacao = StructType([
    StructField("id_notificacao", StringType()),
    StructField("dt_notificacao", TimestampType()),
    StructField("uf_residencia", StringType()),
    StructField("regiao_residencia", StringType()),
    StructField("classificacao_final", StringType()),
    StructField("caso_grave", BooleanType()),
])

notificacoes_stream = (
    spark.readStream
    .schema(esquema_notificacao)
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .json(str(pasta_entrada))
)

print("É um DataFrame de streaming?", notificacoes_stream.isStreaming)
notificacoes_stream.printSchema()

É um DataFrame de streaming? True
root
 |-- id_notificacao: string (nullable = true)
 |-- dt_notificacao: timestamp (nullable = true)
 |-- uf_residencia: string (nullable = true)
 |-- regiao_residencia: string (nullable = true)
 |-- classificacao_final: string (nullable = true)
 |-- caso_grave: boolean (nullable = true)



## 10. Aplicação de janelas temporais

- Utilizar uma janela **Tumbling** (intervalos sem sobreposição) ou **Sliding** (intervalos que podem se sobrepor).
- Informar o tipo e a duração da janela e o deslocamento, quando aplicável.
- Implementar uma agregação coerente com a análise, como contagem, soma ou média.
- Explicar brevemente o que será calculado em cada intervalo.

| Item | Escolha |
|---|---|
| Tipo de janela | **Tumbling** (intervalos consecutivos, sem sobreposição) |
| Duração | **1 dia** |
| Deslocamento | Não se aplica: na tumbling, cada janela começa onde a anterior termina, e cada notificação pertence a uma única janela |
| Coluna de tempo | `dt_notificacao` (tempo do evento, não o horário de chegada do arquivo) |
| Agrupamento | Janela × `regiao_residencia` |
| Agregações | `count(*)` → **notificações** no dia; `sum(caso_grave)` → **casos graves** no dia |

Em cada intervalo, o Spark calcula **quantas notificações de cada região aconteceram naquele dia e quantas delas são graves**. A tumbling foi escolhida porque a vigilância conta casos por dia de notificação: cada caso deve entrar em um único dia, e somar os dias deve dar o total. Numa sliding, o mesmo caso apareceria em várias janelas.

A saída é uma tabela em memória (`memory`), consultada no próprio notebook, em modo `complete`: a cada lote a tabela inteira é recalculada, o que permite ver tanto as janelas novas quanto as antigas que mudaram.

In [20]:
notificacoes_por_janela = (
    notificacoes_stream
    .groupBy(window("dt_notificacao", "1 day").alias("janela"), "regiao_residencia")
    .agg(
        count("*").alias("notificacoes"),
        soma(col("caso_grave").cast("int")).alias("casos_graves"),
    )
)

consulta = (
    notificacoes_por_janela.writeStream
    .format("memory")
    .queryName("notificacoes_por_janela")
    .outputMode("complete")
    .option("checkpointLocation", str(pasta_checkpoint))
    .start()
)


def mostrar_janelas():
    resultado = spark.table("notificacoes_por_janela")
    (
        resultado.select(
            date_format("janela.start", "dd/MM HH:mm").alias("início"),
            date_format("janela.end", "dd/MM HH:mm").alias("fim"),
            col("regiao_residencia").alias("região"),
            "notificacoes",
            "casos_graves",
        )
        .orderBy("início", "região")
        .show(truncate=False)
    )
    total = resultado.agg(soma("notificacoes")).first()[0] or 0
    print(f"Total de notificações processadas até agora: {total}")


print("Consulta ativa?", consulta.isActive)
print("Arquivos na entrada:", len(list(pasta_entrada.iterdir())))
mostrar_janelas()

Consulta ativa? True
Arquivos na entrada: 0
+------+---+------+------------+------------+
|início|fim|região|notificacoes|casos_graves|
+------+---+------+------------+------------+
+------+---+------+------------+------------+

Total de notificações processadas até agora: 0


## 11. Resultados e interpretação

Apresentar os resultados do processamento e demonstrar como a análise se comporta com a chegada de novos registros.

Cada célula abaixo publica um lote, espera o Spark processá-lo (`processAllAvailable()`, usado só para facilitar a demonstração: a consulta já está ativa desde o `start()`) e mostra a tabela de resultado atualizada.

### Lote 1: chegam 30 notificações do dia 05/10

**Observe:** surgem as primeiras janelas, todas do dia 05/10 (00:00 a 00:00 do dia seguinte), uma por região que teve notificação.

In [8]:
print(f"Lote 1 publicado: {publicar_lote(1)} notificações")
consulta.processAllAvailable()
mostrar_janelas()

Lote 1 publicado: 30 notificações
+-----------+-----------+------------+------------+------------+
|início     |fim        |região      |notificacoes|casos_graves|
+-----------+-----------+------------+------------+------------+
|05/10 00:00|06/10 00:00|Centro-Oeste|9           |0           |
|05/10 00:00|06/10 00:00|Nordeste    |7           |0           |
|05/10 00:00|06/10 00:00|Norte       |5           |0           |
|05/10 00:00|06/10 00:00|Sudeste     |8           |1           |
|05/10 00:00|06/10 00:00|Sul         |1           |0           |
+-----------+-----------+------------+------------+------------+

Total de notificações processadas até agora: 30


### Lote 2: chegam mais notificações de 05/10 e as primeiras de 06/10

**Observe:** as janelas de 05/10 **crescem** (chegaram mais notificações daquele dia) e surgem as janelas de 06/10. O Spark manteve o estado de cada janela e somou o lote novo ao que já tinha.

In [9]:
print(f"Lote 2 publicado: {publicar_lote(2)} notificações")
consulta.processAllAvailable()
mostrar_janelas()

Lote 2 publicado: 45 notificações
+-----------+-----------+------------+------------+------------+
|início     |fim        |região      |notificacoes|casos_graves|
+-----------+-----------+------------+------------+------------+
|05/10 00:00|06/10 00:00|Centro-Oeste|17          |0           |
|05/10 00:00|06/10 00:00|Nordeste    |10          |0           |
|05/10 00:00|06/10 00:00|Norte       |6           |0           |
|05/10 00:00|06/10 00:00|Sudeste     |16          |1           |
|05/10 00:00|06/10 00:00|Sul         |1           |0           |
|06/10 00:00|07/10 00:00|Centro-Oeste|8           |0           |
|06/10 00:00|07/10 00:00|Nordeste    |6           |1           |
|06/10 00:00|07/10 00:00|Norte       |1           |0           |
|06/10 00:00|07/10 00:00|Sudeste     |10          |0           |
+-----------+-----------+------------+------------+------------+

Total de notificações processadas até agora: 75


### Lote 3: chegam notificações de 06/10 e 07/10

**Observe:** as janelas de 05/10 **não mudam**, porque nenhuma notificação daquele dia chegou; 06/10 cresce e surgem as janelas de 07/10.

In [10]:
print(f"Lote 3 publicado: {publicar_lote(3)} notificações")
consulta.processAllAvailable()
mostrar_janelas()

Lote 3 publicado: 50 notificações
+-----------+-----------+------------+------------+------------+
|início     |fim        |região      |notificacoes|casos_graves|
+-----------+-----------+------------+------------+------------+
|05/10 00:00|06/10 00:00|Centro-Oeste|17          |0           |
|05/10 00:00|06/10 00:00|Nordeste    |10          |0           |
|05/10 00:00|06/10 00:00|Norte       |6           |0           |
|05/10 00:00|06/10 00:00|Sudeste     |16          |1           |
|05/10 00:00|06/10 00:00|Sul         |1           |0           |
|06/10 00:00|07/10 00:00|Centro-Oeste|14          |0           |
|06/10 00:00|07/10 00:00|Nordeste    |9           |1           |
|06/10 00:00|07/10 00:00|Norte       |1           |0           |
|06/10 00:00|07/10 00:00|Sudeste     |16          |0           |
|07/10 00:00|08/10 00:00|Centro-Oeste|12          |1           |
|07/10 00:00|08/10 00:00|Nordeste    |12          |0           |
|07/10 00:00|08/10 00:00|Norte       |2           |0    

### Lote 4: chegam notificações atrasadas do dia 05/10

Na vigilância real é comum uma notificação ser digitada dias depois. Este lote chega por último, mas traz notificações do **dia 05/10**.

**Observe:** as janelas de **05/10** voltam a crescer, mesmo depois de já existirem janelas de 06 e 07/10. Isso só acontece porque a janela usa o **tempo do evento** (`dt_notificacao`) e não a hora em que o arquivo chegou.

In [11]:
print(f"Lote 4 publicado: {publicar_lote(4)} notificações (atrasadas, do dia 05/10)")
consulta.processAllAvailable()
mostrar_janelas()

Lote 4 publicado: 10 notificações (atrasadas, do dia 05/10)
+-----------+-----------+------------+------------+------------+
|início     |fim        |região      |notificacoes|casos_graves|
+-----------+-----------+------------+------------+------------+
|05/10 00:00|06/10 00:00|Centro-Oeste|21          |0           |
|05/10 00:00|06/10 00:00|Nordeste    |14          |0           |
|05/10 00:00|06/10 00:00|Norte       |6           |0           |
|05/10 00:00|06/10 00:00|Sudeste     |18          |1           |
|05/10 00:00|06/10 00:00|Sul         |1           |0           |
|06/10 00:00|07/10 00:00|Centro-Oeste|14          |0           |
|06/10 00:00|07/10 00:00|Nordeste    |9           |1           |
|06/10 00:00|07/10 00:00|Norte       |1           |0           |
|06/10 00:00|07/10 00:00|Sudeste     |16          |0           |
|07/10 00:00|08/10 00:00|Centro-Oeste|12          |1           |
|07/10 00:00|08/10 00:00|Nordeste    |12          |0           |
|07/10 00:00|08/10 00:00|Norte

### Conferência: streaming × processamento em lote

Se o streaming está correto, o resultado final precisa ser igual ao de calcular a mesma agregação de uma vez sobre todos os arquivos que chegaram (`spark.read`, sem streaming).

In [12]:
todos_os_arquivos = spark.read.schema(esquema_notificacao).option("timestampFormat", "yyyy-MM-dd HH:mm:ss").json(str(pasta_entrada))
resultado_lote = (
    todos_os_arquivos
    .groupBy(window("dt_notificacao", "1 day").alias("janela"), "regiao_residencia")
    .agg(count("*").alias("notificacoes"), soma(col("caso_grave").cast("int")).alias("casos_graves"))
)
resultado_stream = spark.table("notificacoes_por_janela")

so_no_stream = resultado_stream.exceptAll(resultado_lote).count()
so_no_lote = resultado_lote.exceptAll(resultado_stream).count()
total_publicado = sum(len(eventos) for eventos in eventos_por_lote.values())
total_processado = resultado_stream.agg(soma("notificacoes")).first()[0]

print("CONFERÊNCIA")
print("─" * 50)
print(f"Notificações publicadas:        {total_publicado}")
print(f"Notificações processadas:       {total_processado}")
print(f"Linhas só no streaming:         {so_no_stream}")
print(f"Linhas só no cálculo em lote:   {so_no_lote}")
assert total_publicado == total_processado and so_no_stream == 0 and so_no_lote == 0
print("✓ O streaming chegou exatamente ao mesmo resultado do processamento em lote.")

print("\nResumo por dia (soma das regiões):")
(
    resultado_stream.groupBy(date_format("janela.start", "dd/MM").alias("dia"))
    .agg(soma("notificacoes").alias("notificacoes"), soma("casos_graves").alias("casos_graves"))
    .orderBy("dia")
    .show()
)

CONFERÊNCIA
──────────────────────────────────────────────────
Notificações publicadas:        135
Notificações processadas:       135
Linhas só no streaming:         0
Linhas só no cálculo em lote:   0
✓ O streaming chegou exatamente ao mesmo resultado do processamento em lote.

Resumo por dia (soma das regiões):
+-----+------------+------------+
|  dia|notificacoes|casos_graves|
+-----+------------+------------+
|05/10|          60|           1|
|06/10|          40|           1|
|07/10|          35|           1|
+-----+------------+------------+



**Conclusão:**

A cada lote, a tabela de resultado mudou apenas nas janelas que receberam notificações novas:

| Lote | O que chegou | O que mudou na tabela |
|---|---|---|
| 1 | 30 notificações de 05/10 | Surgiram 5 janelas de 05/10, uma por região (Centro-Oeste 9, Sudeste 8, Nordeste 7, Norte 5, Sul 1) |
| 2 | 20 de 05/10 e 25 de 06/10 | As janelas de 05/10 cresceram (Centro-Oeste de 9 para 17, Sudeste de 8 para 16) e surgiram 4 janelas de 06/10 |
| 3 | 15 de 06/10 e 35 de 07/10 | 05/10 ficou igual; 06/10 cresceu (Sudeste de 10 para 16, Centro-Oeste de 8 para 14) e surgiram 4 janelas de 07/10 |
| 4 (atrasado) | 10 de 05/10 | As janelas de 05/10 voltaram a mudar (Centro-Oeste de 17 para 21, Nordeste de 10 para 14, Sudeste de 16 para 18) |

Três comportamentos ficaram visíveis:

- **Estado:** o Spark guardou a contagem de cada janela entre um lote e outro e somou os novos registros ao que já tinha, sem reprocessar os arquivos antigos.
- **Tempo do evento:** as notificações atrasadas do lote 4 entraram na janela de 05/10, e não na do dia em que chegaram, porque a janela usa `dt_notificacao`.
- **Correção:** ao final, as 135 notificações publicadas foram todas processadas, distribuídas em 13 janelas (dia × região), e o resultado é idêntico ao do mesmo cálculo feito em lote: 60 notificações em 05/10, 40 em 06/10 e 35 em 07/10, com 1 caso grave em cada dia.

Centro-Oeste e Sudeste concentram a maior parte das notificações simuladas, e os casos graves são raros (3 de 135). Isso reflete o perfil real usado na simulação (as duas regiões somam cerca de 69% dos casos de 2026 e a proporção de graves é de 3,5%), e não uma tendência nova: o objetivo aqui é mostrar o mecanismo, não tirar conclusões sobre os dias simulados.

### Encerrando a consulta

A consulta continuaria ativa, aguardando novos arquivos. Ela é encerrada explicitamente, e as pastas temporárias são removidas.

In [13]:
consulta.stop()
print("Consulta ativa?", consulta.isActive)
spark.catalog.dropTempView("notificacoes_por_janela")
spark.stop()
area_temporaria.cleanup()
print("✓ Consulta encerrada, Spark finalizado e pastas temporárias removidas.")

Consulta ativa? False
✓ Consulta encerrada, Spark finalizado e pastas temporárias removidas.


## 12. Considerações do RA2

- O que foi possível observar com o processamento em streaming?
- Qual foi a contribuição das janelas temporais para a análise?
- Essa abordagem faria sentido em uma aplicação real do projeto? Por quê?


**Respostas:**

**O que foi possível observar:** O resultado é construído de forma incremental: cada lote altera só as janelas afetadas, o Spark mantém o estado das contagens entre os micro-batches e uma notificação atrasada corrige a janela do dia a que pertence. A conferência com o processamento em lote mostrou que o streaming chega ao mesmo resultado, só que atualizado aos poucos.

**Contribuição das janelas temporais:** Sem janela, a consulta mostraria apenas um total acumulado por região, que só cresce e não diz quando os casos aconteceram. A janela tumbling de 1 dia organiza o fluxo contínuo em dias comparáveis entre si, e como cada notificação entra em um único dia, a soma das janelas é igual ao total de notificações recebidas.

**Faria sentido numa aplicação real?** Sim. A notificação de dengue é contínua, e acompanhar por dia e por região as notificações e os casos graves ajudaria a perceber cedo uma região em alta e a planejar leitos. Neste projeto, porém, o streaming foi um **exercício didático**: a fonte pública é um arquivo anual atualizado periodicamente (a versão usada é de 10/10/2026), não um fluxo de eventos, e os registros dos lotes foram simulados. Numa aplicação real seriam necessários uma fonte contínua (por exemplo, o próprio SINAN enviando cada notificação ao ser registrada), um *watermark* para limitar quanto atraso aceitar e o quanto de estado guardar, e uma saída persistente no lugar da tabela em memória.